# Notebook 2 — Feature Engineering

**Goal:** Transform raw CTA snapshots into a rectangular table of numbers that a model can learn from.

## What is feature engineering?

A statistical model sees only numbers. A datetime like `2024-01-15 08:32:00` is meaningless to it — but the number `8` (hour of day) or `1` (is_weekday) *is* meaningful because it captures the pattern: morning rush hour trains run differently.

Feature engineering is the process of asking: **"What information in my data is predictive of the outcome?"** and encoding it as numbers.

### Our three categories of features

| Category | Examples | Why they matter |
|----------|----------|-----------------|
| **Static** | Route, station, stop position | Red Line North Side runs differently from South Side |
| **Temporal** | Hour, day-of-week, is_rush_hour | Patterns repeat by time; model learns "8am Monday is usually late" |
| **Real-time** | minutes_until_arrival, ETA drift | Live signal about *this specific train right now* |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pytz
from sqlalchemy import text

from backend.db.session import SessionLocal
from backend.stations import get_station

sns.set_theme(style='darkgrid')
chicago = pytz.timezone('America/Chicago')

In [ ]:
# Load raw snapshots
with SessionLocal() as db:
    snaps = pd.read_sql(
        text("""
            SELECT s.*, a.delay_minutes
            FROM arrival_snapshots s
            LEFT JOIN actual_arrivals a
              ON a.run_number = s.run_number
             AND a.station_id = s.station_id
             AND a.actual_arrival_time BETWEEN s.snapshot_time AND s.snapshot_time + INTERVAL '2 hours'
            WHERE s.arr_t IS NOT NULL
            ORDER BY s.snapshot_time
            LIMIT 500000
        """),
        db.bind,
        parse_dates=['snapshot_time', 'prdt', 'arr_t']
    )

print(f'{len(snaps):,} rows loaded')
snaps.head(2)

## Feature 1: `minutes_until_arrival`

This is the single most important real-time feature. It answers: *"How far away is this train right now?"*

Trains 0–2 minutes away have very stable predictions. Trains 15+ minutes away have high uncertainty. The model should learn to be more confident about near-term predictions.

**Formula:** `max(0, (arr_t - snapshot_time).total_seconds() / 60)`

We use `snapshot_time` not `prdt` because `snapshot_time` is when *we measured*, which is what we'll know at prediction time.

In [ ]:
snaps['minutes_until'] = (
    (snaps['arr_t'] - snaps['snapshot_time']).dt.total_seconds() / 60
).clip(0)

# How does prediction error (proxy) relate to time horizon?
# We use std of delay_minutes bucketed by horizon as a proxy for uncertainty.
snaps['horizon_bucket'] = pd.cut(
    snaps['minutes_until'],
    bins=[0, 2, 5, 10, 15, 20, 30],
    labels=['0-2', '2-5', '5-10', '10-15', '15-20', '20-30']
)

if snaps['delay_minutes'].notna().sum() > 100:
    uncertainty = snaps.groupby('horizon_bucket')['delay_minutes'].std()
    fig, ax = plt.subplots(figsize=(8, 3))
    uncertainty.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Std dev of actual delay by prediction horizon')
    ax.set_ylabel('Std dev (minutes)')
    ax.set_xlabel('Minutes until arrival at snapshot time')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    print('Wider bars = more uncertainty at longer horizons (expected)')

## Feature 2: Temporal features

Time patterns are the backbone of transit modeling. Trains don't behave randomly — they follow weekly rhythms. We encode:

- `hour_of_day` (0–23): captures within-day patterns
- `day_of_week` (0=Monday … 6=Sunday): weekday vs weekend is huge
- `is_weekend` (bool): often more informative than raw day number
- `is_peak_am` (7–9am weekday), `is_peak_pm` (4–7pm weekday): explicit rush-hour flags

**Why not just use the raw timestamp?** Because a model trained on timestamps from Jan–March won't know what to do with April timestamps. Hour of day *generalises*; a specific timestamp doesn't.

In [ ]:
snaps['local_dt'] = snaps['snapshot_time'].dt.tz_convert(chicago)
snaps['hour']       = snaps['local_dt'].dt.hour
snaps['dow']        = snaps['local_dt'].dt.dayofweek   # 0=Mon, 6=Sun
snaps['is_weekend'] = snaps['dow'] >= 5
snaps['is_peak_am'] = (snaps['dow'] < 5) & snaps['hour'].between(7, 8)
snaps['is_peak_pm'] = (snaps['dow'] < 5) & snaps['hour'].between(16, 18)

if snaps['delay_minutes'].notna().sum() > 100:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Mean delay by hour
    snaps.groupby('hour')['delay_minutes'].mean().plot(
        ax=axes[0], marker='o', color='steelblue'
    )
    axes[0].set_title('Mean delay by hour of day (Chicago time)')
    axes[0].set_xlabel('Hour')
    axes[0].set_ylabel('Mean delay (minutes)')
    axes[0].axhline(0, color='white', linestyle='--', alpha=0.5)

    # Mean delay by day of week
    day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    snaps.groupby('dow')['delay_minutes'].mean().rename(
        index=dict(enumerate(day_labels))
    ).plot(kind='bar', ax=axes[1], color='steelblue')
    axes[1].set_title('Mean delay by day of week')
    axes[1].set_xlabel('')
    axes[1].set_ylabel('Mean delay (minutes)')
    plt.xticks(rotation=0)

    plt.tight_layout()
    plt.show()

## Feature 3: ETA drift (delta features)

This is a subtle but powerful feature. If a train's predicted arrival time keeps slipping later and later across consecutive polls, that's a strong signal it's falling behind.

We compute:
- `eta_delta_1`: change in `minutes_until` between *this* snapshot and the *previous* one for the same (run, station)
- `eta_delta_2`: change vs two snapshots ago

**Intuition:**
- If a train is running perfectly on schedule, `minutes_until` should decrease by ~0.4 min (25 s ÷ 60) between polls
- If `eta_delta_1` > 0 (the ETA is *increasing* between polls), that's a bad sign
- If `eta_delta_1` is persistently large and positive, the train is likely going to be late

In [ ]:
snaps = snaps.sort_values(['run_number', 'station_id', 'snapshot_time'])

# Shift within (run, station) group to get previous ETA values
grp = snaps.groupby(['run_number', 'station_id'])['minutes_until']
snaps['eta_delta_1'] = snaps['minutes_until'] - grp.shift(1)
snaps['eta_delta_2'] = snaps['minutes_until'] - grp.shift(2)

print('ETA delta_1 stats (expected ~-0.4 for on-time trains):')
print(snaps['eta_delta_1'].describe())

if snaps['delay_minutes'].notna().sum() > 100:
    # Does a positive delta_1 predict lateness?
    fig, ax = plt.subplots(figsize=(8, 4))
    sample = snaps.dropna(subset=['eta_delta_1', 'delay_minutes']).sample(
        min(5000, len(snaps)), random_state=42
    )
    ax.scatter(
        sample['eta_delta_1'].clip(-5, 10),
        sample['delay_minutes'].clip(-10, 20),
        alpha=0.15, s=8, color='steelblue'
    )
    ax.set_xlabel('ETA delta (min) — positive = ETA slipping later')
    ax.set_ylabel('Actual delay (minutes)')
    ax.set_title('ETA drift vs actual delay')
    ax.axhline(0, color='white', linestyle='--', alpha=0.5)
    ax.axvline(0, color='white', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

## Feature 4: Static features — route, station, stop sequence

Some stations are chronically delayed. Some legs of the line are slower. We encode:

- `is_red_line` / `is_blue_line`: the two routes behave differently
- `stop_sequence`: where this station falls along the line (0–31). Trains accumulate delay along the route, so later stops tend to see larger delays.
- `direction_code`: northbound vs southbound (or O'Hare vs Forest Park bound)

**Why not station name as a string?** Tree-based models (XGBoost) can handle integers, but a 32-category string would need one-hot encoding (32 extra columns). Stop sequence is a more compact and generalizable encoding.

In [ ]:
snaps['is_red']  = (snaps['route'] == 'Red').astype(int)
snaps['is_blue'] = (snaps['route'] == 'Blue').astype(int)

def get_stop_seq(station_id):
    try:
        s = get_station(int(station_id))
        return s.stop_sequence if s else None
    except (ValueError, TypeError):
        return None

snaps['stop_sequence'] = snaps['station_id'].apply(get_stop_seq)

if snaps['delay_minutes'].notna().sum() > 100:
    # Does delay accumulate along the line?
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, route in zip(axes, ['Red', 'Blue']):
        sub = snaps[(snaps['route'] == route)].dropna(subset=['stop_sequence', 'delay_minutes'])
        sub.groupby('stop_sequence')['delay_minutes'].mean().plot(
            ax=ax, marker='o', color='#c60c30' if route == 'Red' else '#00a1de'
        )
        ax.set_title(f'{route} Line — mean delay by stop sequence')
        ax.set_xlabel('Stop sequence (0=terminus)')
        ax.set_ylabel('Mean delay (minutes)')
        ax.axhline(0, color='white', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

## Feature 5: CTA quality flags

CTA provides three boolean flags:
- `is_scheduled` — the train is running on schedule-only tracking (no live GPS). These predictions are *less reliable*.
- `is_delayed` — CTA itself has flagged the train as delayed.
- `is_faulty` — the prediction data is potentially erroneous.

These are strong signals. A train already flagged delayed by CTA is almost certain to show up late in our data.

In [ ]:
if snaps['delay_minutes'].notna().sum() > 100:
    print('Mean actual delay by CTA delayed flag:')
    print(snaps.groupby('is_delayed')['delay_minutes'].agg(['mean', 'median', 'count']))
    print()
    print('Mean actual delay when is_scheduled:')
    print(snaps.groupby('is_scheduled')['delay_minutes'].agg(['mean', 'median', 'count']))

## Assembling the feature matrix

Now we put all features together into one DataFrame with the target variable. This is what we'll feed to the models.

In [ ]:
feature_cols = [
    'is_red', 'is_blue',
    'stop_sequence', 'direction',
    'hour', 'dow', 'is_weekend', 'is_peak_am', 'is_peak_pm',
    'minutes_until',
    'is_scheduled', 'is_delayed', 'is_faulty',
    'eta_delta_1', 'eta_delta_2',
]

# Cast booleans to int (models prefer numbers)
for col in ['is_weekend', 'is_peak_am', 'is_peak_pm', 'is_scheduled', 'is_delayed', 'is_faulty']:
    if col in snaps.columns:
        snaps[col] = snaps[col].astype(int)

# direction: coerce to numeric (CTA direction code 1 or 5)
snaps['direction'] = pd.to_numeric(snaps['direction'], errors='coerce').fillna(0).astype(int)

features = snaps[feature_cols + ['delay_minutes', 'snapshot_time', 'route', 'station_id']].dropna(
    subset=['delay_minutes']  # only keep labelled rows
)

print(f'Labelled rows (have actual delay): {len(features):,}')
print('\nFeature matrix preview:')
features[feature_cols].head(3)

In [ ]:
# Correlation with target — tells us which features are most linearly related to delay
corr = features[feature_cols].corrwith(features['delay_minutes']).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
corr.plot(kind='barh', ax=ax, color=corr.apply(lambda x: 'tomato' if x > 0 else 'steelblue'))
ax.set_title('Linear correlation of each feature with delay_minutes')
ax.set_xlabel('Pearson correlation')
ax.axvline(0, color='white', alpha=0.5)
plt.tight_layout()
plt.show()

print("\nNote: low correlation doesn't mean useless — tree models can use non-linear relationships.")

In [ ]:
# Save for use in notebook 3
features.to_parquet('../data/features.parquet', index=False)
print('Saved to data/features.parquet')

## Key takeaways

1. **`minutes_until_arrival`** is the strongest single predictor — the closer the train, the better CTA's estimate, the smaller our job.
2. **ETA drift** (`eta_delta_1`) captures momentum — a train whose ETA is *increasing* between polls is likely going to be late.
3. **Temporal features** encode recurring patterns. Rush-hour trains behave differently from off-peak.
4. **CTA's own `is_delayed` flag** is a very informative feature — if they already know it's delayed, so does our model.

**What we're *not* yet using:**
- Headway to the preceding/following train
- Weather (external data source)
- Service alerts

➡️ **Next:** [03_baseline_models.ipynb](03_baseline_models.ipynb)